# Creating a Failure-Aware Placement Algorithm

This tutorial demonstrates how a placement algorithm can use the network topology to limit how many users lose their applications when central sites fail, and what that choice costs when other sites fail.

Here, a site is a base station together with its network switch and edge servers, and we assume that they all fail together. Users whose services ran there lose their applications, and so can users whose traffic crossed that switch. Since EdgeSimPy's `Topology` is a NetworkX graph, we can measure how central each site is and keep services away from central sites while each user stays within its delay SLA.

Let's start by importing the EdgeSimPy modules:

In [1]:
try:
    # Importing EdgeSimPy components
    from edge_sim_py import *
    import networkx as nx
    import msgpack

    # Importing Matplotlib, Pandas, and NumPy for logs parsing and visualization
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

except ModuleNotFoundError:
    # Downloading EdgeSimPy binaries from GitHub (the "-q" parameter suppresses Pip's output. You can check the full logs by removing it)
    %pip install -q git+https://github.com/EdgeSimPy/EdgeSimPy.git

    # Downloading Pandas, NumPy, and Matplotlib (Pandas and NumPy are used below for the result tables)
    %pip install -q pandas
    %pip install -q numpy
    %pip install -q matplotlib

    # Importing EdgeSimPy components and its built-in libraries (NetworkX and MessagePack)
    from edge_sim_py import *
    import networkx as nx
    import msgpack

    # Importing Matplotlib, Pandas, and NumPy for logs parsing and visualization
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

## Loading the Scenario

The sample datasets are too small for the choice of host to matter much when a site fails, so we use a larger dataset built with the functions of EdgeSimPy's dataset generator: 49 base stations on a hexagonal grid map connected by a Barabási-Albert topology, 16 edge servers of the generator's six server models, and 24 users, each accessing a single-service application with a delay SLA of 25 or 45 ms. Every network link and every wireless connection adds 5 ms of delay.

The function below runs the simulation until all services are provisioned. It keeps users at their starting base stations (the dataset gives them the Pathway mobility model), so that every algorithm is evaluated with users in the same places. It can also change the users' delay SLAs, which we use at the end of the tutorial.

In [2]:
# Dataset used throughout this tutorial
DATASET = "https://raw.githubusercontent.com/EdgeSimPy/edgesimpy-tutorials/master/datasets/sample_dataset3.json"


def stay_in_place(user: object):
    # Mobility model that keeps the user at its current position
    user.coordinates_trace.append(user.coordinates)


def stopping_criterion(model: object):
    # The simulation stops once every service has been provisioned in an edge server
    return all(service.server != None for service in Service.all())


def run_simulation(algorithm, new_delay_slas=None):
    # Creating a Simulator object that uses the given placement algorithm
    simulator = Simulator(
        tick_duration=1,
        tick_unit="seconds",
        stopping_criterion=stopping_criterion,
        resource_management_algorithm=algorithm,
    )

    # Loading the dataset
    simulator.initialize(input_file=DATASET)

    for user in User.all():
        # Keeping the user at its starting base station
        user.mobility_model = stay_in_place

        # Optionally replacing each delay SLA (e.g., {25: 15, 45: 25} turns 25 ms SLAs into 15 ms and 45 ms SLAs into 25 ms)
        if new_delay_slas:
            for application_id, delay_sla in user.delay_slas.items():
                user.delay_slas[application_id] = new_delay_slas[delay_sla]

    # Executing the simulation
    simulator.run_model()

## Measuring How Central Each Site Is

We use betweenness centrality: the share of shortest paths between pairs of network switches that pass through a given switch. As EdgeSimPy routes traffic along the paths with the lowest delay, we use link delays as distances. Let's load the dataset and check how central the sites that host edge servers are:

In [3]:
def site_centrality():
    # Betweenness centrality of each network switch, using link delays as distances
    return nx.betweenness_centrality(Topology.first(), weight="delay")


# Loading the dataset to inspect the infrastructure
simulator = Simulator()
simulator.initialize(input_file=DATASET)

centrality = site_centrality()
server_sites = sorted(
    {server.base_station for server in EdgeServer.all()}, key=lambda base_station: (-centrality[base_station.network_switch], base_station.id)
)

for base_station in server_sites:
    servers = ", ".join(f"{server.model_name} ({server.cpu} CPUs)" for server in base_station.edge_servers)
    print(f"{base_station}. Centrality: {centrality[base_station.network_switch]:.3f}. Edge servers: {servers}")

BaseStation_1. Centrality: 0.244. Edge servers: NVIDIA Jetson TX2 (4 CPUs)
BaseStation_9. Centrality: 0.095. Edge servers: E5430 (8 CPUs)
BaseStation_2. Centrality: 0.082. Edge servers: E5430 (8 CPUs)
BaseStation_5. Centrality: 0.077. Edge servers: E5645 (12 CPUs)
BaseStation_7. Centrality: 0.047. Edge servers: NVIDIA Jetson Nano (4 CPUs)
BaseStation_8. Centrality: 0.044. Edge servers: NVIDIA Jetson Nano (4 CPUs)
BaseStation_28. Centrality: 0.029. Edge servers: E5645 (12 CPUs)
BaseStation_25. Centrality: 0.019. Edge servers: E5645 (12 CPUs)
BaseStation_29. Centrality: 0.017. Edge servers: E5430 (8 CPUs)
BaseStation_14. Centrality: 0.009. Edge servers: NVIDIA Jetson TX2 (4 CPUs)
BaseStation_41. Centrality: 0.008. Edge servers: E5507 (8 CPUs)
BaseStation_37. Centrality: 0.003. Edge servers: E5507 (8 CPUs)
BaseStation_44. Centrality: 0.002. Edge servers: Raspberry Pi 4 (4 CPUs)
BaseStation_17. Centrality: 0.002. Edge servers: NVIDIA Jetson TX2 (4 CPUs)
BaseStation_31. Centrality: 0.002. E

## Estimating the Delay Each Host Would Give a User

EdgeSimPy computes the delay perceived by a user as the wireless delay of the user's base station plus the delay of the shortest path between that base station and the base station hosting the service. We can estimate that delay for any candidate host before provisioning the service:

In [4]:
def user_delay(user, edge_server):
    # Delay between the user and its base station
    delay = user.base_station.wireless_delay

    # Delay of the shortest path between the user's base station and the edge server's base station
    delay += nx.shortest_path_length(
        G=Topology.first(),
        source=user.base_station.network_switch,
        target=edge_server.base_station.network_switch,
        weight="delay",
    )

    return delay

## Implementing the Placement Algorithm

Our algorithm provisions each service on the least central edge server that has enough resources to host it and keeps its user within the delay SLA. If no edge server meets the SLA, the algorithm falls back to the one with the lowest delay.

In [5]:
def least_central_placement(parameters):
    centrality = site_centrality()

    for service in Service.all():
        # We only place services that are not hosted or being provisioned yet
        if service.server == None and not service.being_provisioned:

            # Each application in this dataset has a single service and a single user
            application = service.application
            user = application.users[0]
            delay_sla = user.delay_slas[str(application.id)]

            # Edge servers with enough free resources to host the service
            candidates = [edge_server for edge_server in EdgeServer.all() if edge_server.has_capacity_to_host(service=service)]

            # Edge servers that would keep the user within the delay SLA
            within_sla = [edge_server for edge_server in candidates if user_delay(user, edge_server) <= delay_sla]

            if within_sla:
                # Choosing the least central host (ties go to the host with the lowest delay)
                target = min(within_sla, key=lambda edge_server: (centrality[edge_server.base_station.network_switch], user_delay(user, edge_server)))
            elif candidates:
                # No host meets the SLA, so we choose the one with the lowest delay
                target = min(candidates, key=lambda edge_server: user_delay(user, edge_server))
            else:
                continue

            service.provision(target_server=target)

We compare our algorithm with two algorithms that do not consider how central a site is: the First-Fit heuristic from the "Creating a Placement Algorithm" tutorial, and an algorithm that provisions each service on the edge server closest to its user.

In [6]:
def first_fit_placement(parameters):
    for service in Service.all():
        if service.server == None and not service.being_provisioned:
            # Provisioning the service on the first edge server with enough free resources
            for edge_server in EdgeServer.all():
                if edge_server.has_capacity_to_host(service=service):
                    service.provision(target_server=edge_server)
                    break


def closest_server_placement(parameters):
    for service in Service.all():
        if service.server == None and not service.being_provisioned:
            # Provisioning the service on the edge server with enough free resources and the lowest delay to its user
            user = service.application.users[0]
            candidates = [edge_server for edge_server in EdgeServer.all() if edge_server.has_capacity_to_host(service=service)]
            if candidates:
                service.provision(target_server=min(candidates, key=lambda edge_server: user_delay(user, edge_server)))

## Measuring What a Failure Would Break

We measure each failure on the topology once all services are placed: we remove the failed sites' switches from a copy of the topology and check every user outside those sites. A user loses their application if the service was hosted at a failed site, if they can no longer reach it, or if the detour breaks a delay SLA that was met before the failure. Users attached to a failed site are left out, because no placement could have served them.

In [7]:
def users_affected(failed_sites):
    # Copying the topology into a plain NetworkX graph (calling Topology.copy() would create a second Topology object)
    # and removing the network switches of the failed sites
    surviving_topology = nx.Graph(Topology.first())
    surviving_topology.remove_nodes_from([base_station.network_switch for base_station in failed_sites])

    affected_users = 0
    for user in User.all():
        # Users attached to a failed site could not be served by any placement
        if user.base_station in failed_sites:
            continue

        application = user.applications[0]
        delay_sla = user.delay_slas[str(application.id)]
        host_site = application.services[0].server.base_station

        if host_site in failed_sites:
            # The service was hosted at a failed site
            affected_users += 1
        elif not nx.has_path(surviving_topology, user.base_station.network_switch, host_site.network_switch):
            # The user can no longer reach the service
            affected_users += 1
        else:
            # The user can still reach the service, but the detour may exceed the delay SLA
            delay_before = user_delay(user, application.services[0].server)
            delay_after = user.base_station.wireless_delay + nx.shortest_path_length(
                G=surviving_topology,
                source=user.base_station.network_switch,
                target=host_site.network_switch,
                weight="delay",
            )
            if delay_before <= delay_sla < delay_after:
                affected_users += 1

    return affected_users

We look at three kinds of failure:
- The most central 20% of the sites hosting edge servers (3 of the 16 sites) fail together.
- A site fails together with the sites linked to it. We average this over all 49 sites, so the failed sites are not chosen by centrality.
- The single site hosting edge servers whose failure affects the most users.

In [8]:
def evaluate_failures():
    centrality = site_centrality()
    topology = Topology.first()

    # Sites hosting edge servers, from the most to the least central
    server_sites = sorted(
        {server.base_station for server in EdgeServer.all()}, key=lambda base_station: (-centrality[base_station.network_switch], base_station.id)
    )
    most_central_sites = set(server_sites[: round(0.2 * len(server_sites))])

    # Each base station together with the base stations linked to it
    neighborhoods = []
    for base_station in BaseStation.all():
        linked_sites = {switch.base_station for switch in topology.neighbors(base_station.network_switch)}
        neighborhoods.append({base_station} | linked_sites)

    return {
        "Central failure": users_affected(most_central_sites),
        "Neighborhood failure (mean)": round(float(np.mean([users_affected(neighborhood) for neighborhood in neighborhoods])), 2),
        "Worst single site": max(users_affected({site}) for site in server_sites),
        "Edge servers used": len({service.server for service in Service.all()}),
        "Users within SLA": sum(
            user_delay(user, user.applications[0].services[0].server) <= user.delay_slas[str(user.applications[0].id)] for user in User.all()
        ),
    }

## Comparing the Algorithms

Let's run the simulation once with each algorithm. Each failure column gives the number of users affected:

In [9]:
algorithms = {
    "First-Fit": first_fit_placement,
    "Closest server": closest_server_placement,
    "Least central": least_central_placement,
}


def compare_algorithms(new_delay_slas=None):
    results = {}
    for name, algorithm in algorithms.items():
        run_simulation(algorithm, new_delay_slas)
        results[name] = evaluate_failures()
    return pd.DataFrame.from_dict(results, orient="index")


compare_algorithms()

,Central failure,Neighborhood failure (mean),Worst single site,Edge servers used,Users within SLA
First-Fit,7,3.39,8,5,23
Closest server,9,2.18,7,9,24
Least central,2,2.73,5,6,24


The central failure is the one our algorithm targets: it affects 2 users, against 7 with First-Fit and 9 with the closest-server algorithm. Our algorithm also has the smallest worst single-site failure on this map (5 users), which is unusual, as the Discussion shows.

The neighborhood failures are a fairer test, because their sites are not chosen by centrality. There, the closest-server algorithm affects the fewest users (2.18 on average, against 2.73 for our algorithm and 3.39 for First-Fit), as its services sit near their users and are spread over more servers. First-Fit, which ignores delay, also leaves one user outside its SLA.

## Tighter Delay SLAs

On this map, no user is more than 30 ms away from any edge server, so the 45 ms SLA never limits the choice of host and the 25 ms SLA rarely does. Let's turn every 25 ms SLA into 15 ms and every 45 ms SLA into 25 ms:

In [10]:
compare_algorithms(new_delay_slas={25: 15, 45: 25})

,Central failure,Neighborhood failure (mean),Worst single site,Edge servers used,Users within SLA
First-Fit,9,4.06,8,5,16
Closest server,9,2.90,7,9,24
Least central,4,3.29,4,9,24


## Discussion

With tighter SLAs, our algorithm has fewer distant servers to choose from, and the central failure now affects 4 users instead of 2. That is still fewer than the 9 with the other two algorithms, and the closest-server algorithm still does best in the neighborhood failures. First-Fit leaves 8 users outside their SLAs, and its failure counts look better than they are, because a detour can only break an SLA that was met before the failure.

This is one map. Over ten maps built the same way with the dataset's SLAs, the central failure affected 1.8 users on average with our algorithm, 4.7 with First-Fit and 6.8 with the closest-server algorithm. The neighborhood failures affected 2.81 users with our algorithm, 3.25 with First-Fit and 2.09 with the closest-server algorithm. The tutorial's map was the only one where our algorithm had the smallest worst single-site failure. On those maps, our algorithm used 4 to 7 edge servers, where the closest-server algorithm used 9 to 13, so each of its servers hosted more services. With the tighter SLAs, our algorithm had the fewest users affected by the central failure on only 4 of the ten maps (4.8 on average, against 5.4 for First-Fit and 6.9 for the closest-server algorithm), and the closest-server algorithm did best in the neighborhood failures on all ten.

So avoiding central sites protects against failures of those sites, while keeping services close to their users protects against local failures. An algorithm could weigh both, for example by choosing the least central site among the hosts closest to the user.

The dataset was built with the `build_scenario` function of the [edgesimpy-cei](https://github.com/prawalpokharel/edgesimpy-cei) package (seed 1, Barabási-Albert topology, one service per application), which uses the dataset generator's map, topology and server builders and the container images of `sample_dataset2.json`. The package's `examples/failure_aware_tutorial_maps.py` runs the ten-map comparison.